In [1]:
import pandas as pd
training_data_file_path=r"C:\Users\khmam\Desktop\data_sets\ai4i2020.csv"
train_data=pd.read_csv(training_data_file_path)
features=['Type','Air temperature','Process temperature','Rotational speed','Torque','Tool wear']
X=train_data[features]
y_features=['Machine failure','TWF','HDF','PWF','OSF','RNF']
y=train_data[y_features]
train_data.head(20)

,UDI,Product ID,Type,Air temperature,Process temperature,Rotational speed,Torque,Tool wear,Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0
5,6,M14865,M,298.1,308.6,1425,41.9,11,0,0,0,0,0,0
6,7,L47186,L,298.1,308.6,1558,42.4,14,0,0,0,0,0,0
7,8,L47187,L,298.1,308.6,1527,40.2,16,0,0,0,0,0,0
8,9,M14868,M,298.3,308.7,1667,28.6,18,0,0,0,0,0,0
9,10,M14869,M,298.5,309.0,1741,28.0,21,0,0,0,0,0,0


In [2]:
#Checking for empty values
X.isnull().sum()

Type                   0
Air temperature        0
Process temperature    0
Rotational speed       0
Torque                 0
Tool wear              0
dtype: int64

In [3]:
#Splitting data into training and validation sets
from sklearn.model_selection import train_test_split
X_train,X_valid,y_train,y_valid=train_test_split(X,y,train_size=0.8,test_size=0.2)

In [7]:
#Preprocessing numerical data
from sklearn.preprocessing import StandardScaler,MinMaxScaler

scaler=MinMaxScaler()
X_train_scaled=X_train.copy()
X_valid_scaled=X_valid.copy()

numerical_cols=['Air temperature','Process temperature','Rotational speed','Torque','Tool wear']

X_train_scaled[numerical_cols]=scaler.fit_transform(X_train[numerical_cols])
X_valid_scaled[numerical_cols]=scaler.transform(X_valid[numerical_cols])

X_train_scaled.head(20)

,Type,Air temperature,Process temperature,Rotational speed,Torque,Tool wear
4480,H,0.804348,0.580247,0.080305,0.708333,0.079051
6134,L,0.619565,0.654321,0.242087,0.455556,0.501976
3716,L,0.760870,0.703704,0.258499,0.390278,0.193676
8769,M,0.228261,0.358025,0.166471,0.513889,0.778656
6878,H,0.608696,0.679012,0.205158,0.427778,0.355731
1977,M,0.304348,0.246914,0.272567,0.368056,0.573123
4130,L,0.717391,0.580247,0.069168,0.813889,0.086957
1055,L,0.163043,0.259259,0.182884,0.488889,0.482213
5255,M,0.891304,0.901235,0.137749,0.662500,0.304348
1212,L,0.173913,0.271605,0.111958,0.726389,0.434783


In [8]:
from sklearn.preprocessing import OneHotEncoder
import pandas as pd

encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# Fit and transform
oh_train_cols = encoder.fit_transform(X_train_scaled[['Type']])
oh_train_cols_df = pd.DataFrame(oh_train_cols, columns=encoder.get_feature_names_out(['Type']))
oh_train_cols_df.index = X_train_scaled.index  # Preserve the index

# Transform validation data
oh_valid_cols = encoder.transform(X_valid_scaled[['Type']])
oh_valid_cols_df = pd.DataFrame(oh_valid_cols, columns=encoder.get_feature_names_out(['Type']))
oh_valid_cols_df.index = X_valid_scaled.index  # Preserve the index

# Dropping 'Type' feature and concatenating with encoded columns
encoded_X_train = X_train_scaled.drop('Type', axis=1)
encoded_X_train = pd.concat([encoded_X_train, oh_train_cols_df], axis=1)

encoded_X_valid = X_valid_scaled.drop('Type', axis=1)
encoded_X_valid = pd.concat([encoded_X_valid, oh_valid_cols_df], axis=1)


encoded_X_train.head()



,Air temperature,Process temperature,Rotational speed,Torque,Tool wear,Type_H,Type_L,Type_M
4480,0.804348,0.580247,0.080305,0.708333,0.079051,1.0,0.0,0.0
6134,0.619565,0.654321,0.242087,0.455556,0.501976,0.0,1.0,0.0
3716,0.760870,0.703704,0.258499,0.390278,0.193676,0.0,1.0,0.0
8769,0.228261,0.358025,0.166471,0.513889,0.778656,0.0,0.0,1.0
6878,0.608696,0.679012,0.205158,0.427778,0.355731,1.0,0.0,0.0


In [9]:
#Model training and prediction
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier
model=XGBClassifier(learning_rate=0.008,n_estimators=500,eval_metric='logloss')
model.fit(encoded_X_train,y_train)
prediction=model.predict(encoded_X_valid)
print(accuracy_score(prediction,y_valid))

0.98


In [ ]:
#Using SVM for classification
